#Severity Estimation Model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# 1. Load Data
df = pd.read_csv('synthetic_crime_severity_real_zones.csv')

# 2. Create Classes (Low/Medium/High) from the Score
# This allows us to calculate F1 Score and Accuracy
bins = [-1, 33, 66, 101]
labels = ['Low', 'Medium', 'High']
df['Severity_Class'] = pd.cut(df['Harm_Score'], bins=bins, labels=labels)

# 3. Preprocessing
le_enc = LabelEncoder()
df['crime_code'] = le_enc.fit_transform(df['crime'])
df['zone_code'] = le_enc.fit_transform(df['Patrol_Zone_DS'])
df['Hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0)

X = df[['crime_code', 'Hour', 'zone_code', 'victim_age']]
y = le_enc.fit_transform(df['Severity_Class']) # Encode Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Compare Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

print(f"{'Model':<20} | {'Accuracy':<10} | {'F1 Score':<10}")
print("-" * 45)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:<20} | {acc:.4f}     | {f1:.4f}")

Model                | Accuracy   | F1 Score  
---------------------------------------------
Random Forest        | 0.9482     | 0.9483
Gradient Boosting    | 0.9503     | 0.9504
SVM                  | 0.8571     | 0.8538
KNN                  | 0.7598     | 0.7616


##Model Training


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1. LOAD & PREPARE DATA
print("📂 Loading synthetic training data...")
df = pd.read_csv('synthetic_crime_severity_real_zones.csv')

# Encode Categorical Features
le_crime = LabelEncoder()
df['crime_code'] = le_crime.fit_transform(df['crime'])

le_zone = LabelEncoder()
df['zone_code'] = le_zone.fit_transform(df['Patrol_Zone_DS'])

# Feature Engineering: Extract Hour
df['Hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0).astype(int)

# Define Features (X) and Target (y)
X = df[['crime_code', 'Hour', 'zone_code', 'victim_age']]
y = df['Harm_Score']

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 2. TRAIN XGBOOST MODEL

print("\n🚀 Training XGBoost Regressor...")

# Configure XGBoost for Regression
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# 3. EVALUATE PERFORMANCE
predictions = xgb_model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("-" * 40)
print(f" Model Training Complete")
print(f" MSE (Mean Squared Error): {mse:.2f}")
print(f" R2 Score (Accuracy): {r2:.4f} ")
print("-" * 40)

# 4. SAVE THE MODEL & ENCODERS
xgb_model.save_model("xgboost_severity_model.json")
joblib.dump(le_crime, "encoder_crime.pkl")
joblib.dump(le_zone, "encoder_zone.pkl")

print("\n💾 Model and Encoders saved!")
print("   - xgboost_severity_model.json")
print("   - encoder_crime.pkl")
print("   - encoder_zone.pkl")

📂 Loading synthetic training data...

🚀 Training XGBoost Regressor...
----------------------------------------
 Model Training Complete
 MSE (Mean Squared Error): 10.75
 R2 Score (Accuracy): 0.9801 
----------------------------------------

💾 Model and Encoders saved!
   - xgboost_severity_model.json
   - encoder_crime.pkl
   - encoder_zone.pkl
